
# PhishU — Master Notebook (EDA • PCA • Tabular Models • Semantic Baselines)

This notebook orchestrates all analyses and models without changing individual scripts.

**Sections**  
1. Environment & Logging  
2. Data loading (single source of truth)  
3. EDA (src/data_analysis/EDA.py)  
4. PCA / ACP (src/data_analysis/ACP.py)  
5. Tabular models: Logistic Regression, Random Forest, XGBoost (src/models/*.py)  
6. Semantic baselines: TF-IDF+LinearSVC, FastText+LogReg (src/semantic_models/...)  
7. Summary comparison table


## 1) Environment & Logging

In [ ]:
from src.pipeline.logger import init_logging, get_logger, set_seed
init_logging("INFO")  # set "DEBUG" for more verbose logs
log = get_logger("Notebook", "INFO")
set_seed(42)
log.info("Notebook started.")


## 2) Data loading

In [ ]:

# Load from UCI via our shared DataLoader
from src.utils.data_loader import DataLoader

dl = DataLoader(dataset_id=967)  # PhiUSIIL Phishing URL dataset
X_all, y_series, meta = dl.get_xy_as_dataframes()

# Build a single DataFrame
df = X_all.copy()
df["label"] = y_series.astype(int).values  # ensure numeric labels

log.info(f"Loaded from UCI: {df.shape[0]} rows, {df.shape[1]} cols "
         f"(features={X_all.shape[1]})")
df.head(3)

## 3) EDA

In [ ]:

import pandas as pd
from src.data_analysis.EDA import run_eda

eda_out = run_eda(
    df=df,
    label_col="label",
    save_dir="outputs/eda",
    save_fig=True,
    show_fig=False
)

display(pd.Series(eda_out["shape"], index=["rows","cols"]).to_frame("shape"))
display(eda_out["dtypes_counts"].to_frame("count").T)
display(eda_out["label_distribution_pct"].to_frame("pct"))
if eda_out["preview_describe"].shape[0] > 0:
    display(eda_out["preview_describe"])


## 4) PCA / ACP

In [ ]:

from src.data_analysis.ACP import run_pca

pca_out = run_pca(
    df=df,
    label_col="label",
    n_components=0.95,
    save_dir="outputs/pca",
    save_fig=True,
    show_fig=False
)

log.info(f"PCA retained components: {pca_out['n_components_']} "
         f"(cumulative variance={pca_out['explained_variance_ratio'].sum():.3f})")
display(pca_out["components_df"].head(5))
display(pd.DataFrame({
    "PC1_top": pca_out["pc1_top_loadings"],
    "PC2_top": pca_out["pc2_top_loadings"]
}))


## 5) Tabular models

In [ ]:

from src.models.regressionlogistique import run_logistic_regression
from src.models.randomforest import run_random_forest
from src.models.XGboost import run_xgboost
from src.models.NN import run_neural_network

tabular_results = {}
common_features = (
    'URLLength','DomainLength','NoOfSubDomain','IsDomainIP',
    'NoOfLettersInURL','NoOfDegitsInURL','NoOfEqualsInURL',
    'NoOfQMarkInURL','NoOfAmpersandInURL','NoOfOtherSpecialCharsInURL',
    'SpacialCharRatioInURL','TLDLength'
)

log.info("Running Logistic Regression (tabular)...")
logreg_out = run_logistic_regression(
    df=df,
    label_col="label",
    features=common_features,
    sample_n=10000,
    save_dir="outputs/logreg",
    save_fig=True,
    show_fig=False
)
tabular_results["LogisticRegression"] = logreg_out["metrics"]
display(pd.Series(logreg_out["metrics"], name="LogisticRegression"))

log.info("Running Random Forest (tabular)...")
rf_out = run_random_forest(
    df=df,
    label_col="label",
    features=logreg_out["X_columns"],
    sample_n=10000,
    save_dir="outputs/random_forest",
    save_fig=True,
    show_fig=False
)
tabular_results["RandomForest"] = rf_out["metrics"]
display(pd.Series(rf_out["metrics"], name="RandomForest"))

log.info("Running XGBoost (tabular)...")
xgb_out = run_xgboost(
    df=df,
    label_col="label",
    features=logreg_out["X_columns"],
    sample_n=10000,
    save_dir="outputs/xgboost",
    save_fig=True,
    show_fig=False
)
tabular_results["XGBoost"] = xgb_out["metrics"]
display(pd.Series(xgb_out["metrics"], name="XGBoost"))

log.info("Running Neural Network (tabular)...")
nn_out = run_neural_network(
    df=df,
    label_col="label",
    features=logreg_out["X_columns"],
    early_stop=True,
    save_dir="outputs/neural_network",
    save_fig=True,
    show_fig=False
)
tabular_results["NeuralNetwork"] = nn_out["metrics"]
display(pd.Series(nn_out["metrics"], name="NeuralNetwork"))


## 6) Semantic baselines (TF-IDF+LinearSVC, FastText+LogReg)

In [ ]:

from src.semantic_models.phishing_url_semantic_baselines import (
    train_and_compare_semantic_baselines, PhiUSIILSpec
)

sem_out = train_and_compare_semantic_baselines(
    spec=PhiUSIILSpec(dataset_id=967),
    test_size=0.2,
    random_state=42,
    run_char_tfidf=True,
    run_fasttext=True,
    fasttext_fast_mode=True
)

def _extract_metrics(res_dict):
    out = {}
    for k, v in res_dict.items():
        out[k] = {
            "roc_auc": float(v.get("roc_auc", float("nan"))),
            "pr_auc": float(v.get("pr_auc", float("nan")))
        }
    return out

sem_metrics = _extract_metrics(sem_out)
sem_metrics


## 7) Summary comparison table

In [ ]:

import pandas as pd
tab_df = pd.DataFrame(tabular_results).T[["accuracy","precision","recall","f1","roc_auc"]]
sem_df = pd.DataFrame(sem_metrics).T[["roc_auc","pr_auc"]]

display(tab_df.style.format("{:.4f}").set_caption("Tabular models"))
display(sem_df.style.format("{:.4f}").set_caption("Semantic baselines"))
